# Prototype: xgb_tfidf

Reproduction check for `detectors/xgb_tfidf.py`. Not run by the pipeline: upload to Colab (File > Upload notebook).

**Runtime:** CPU runtime is enough.

**Decides:** Reproduce Oedingen et al. Table 3 (XGB + TF-IDF): accuracy 98.28, recall 98.70, AUC 99.84, within 1 pp. Cell 4 shows what the shipped model was trained on; cell 2 is the future detectors/xgb_tfidf.py.

Run top to bottom, then paste the printed comparison back into the chat.

In [ ]:
# %% 1. Install (fresh runtime). Pins come from the repo's app/cog.yaml; requirements.txt omits xgboost.
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "xgboost==2.1.0", "scikit-learn==1.5.1", "tiktoken==0.7.0"], check=True)

In [ ]:
# %% 2. Detector module. This cell is detectors/xgb_tfidf.py verbatim.
import functools, hashlib, importlib, io, os, pickle, urllib.request
import numpy as np, tiktoken, xgboost as xgb

_RAW = ("https://raw.githubusercontent.com/MarcOedingen/ChatGPT-Code-Detection/"
        "4aa771dc59a08b2233f75486235374f81b0a5bb4/")
_SHA256 = {
    "Models/Vectorizer_TFIDF.pkl": "62c9cc09caa85a4e82f74293993c63f9dbd046164834addedca09f9aa45ac8b2",
    "app/models/XGB_TFIDF.json": "b3ba52bd67849cc90588c65003059fbd6775672e1d4f39c036506075cac215c0",
}
_CACHE = os.path.expanduser("~/.cache/xgb_tfidf")


def _fetch(rel):
    path = os.path.join(_CACHE, rel)
    if not os.path.exists(path):
        os.makedirs(os.path.dirname(path), exist_ok=True)
        urllib.request.urlretrieve(_RAW + rel, path)
    data = open(path, "rb").read()
    if hashlib.sha256(data).hexdigest() != _SHA256[rel]:
        raise RuntimeError(f"checksum mismatch: {path}")
    return path, data


class _State:
    def __setstate__(self, state):
        self.__dict__.update(state)


class _SafeUnpickler(pickle.Unpickler):
    _NUMPY = {("numpy", "dtype"), ("numpy", "float64"), ("numpy", "ndarray"),
              ("numpy.core.multiarray", "_reconstruct"), ("numpy.core.multiarray", "scalar")}

    def find_class(self, module, name):
        if module == "sklearn.feature_extraction.text":
            return _State
        if (module, name) in self._NUMPY:
            return getattr(importlib.import_module(module), name)
        raise pickle.UnpicklingError(f"blocked global {module}.{name}")


@functools.cache
def _model():
    vec = _SafeUnpickler(io.BytesIO(_fetch("Models/Vectorizer_TFIDF.pkl")[1])).load()
    vocab = {int(tok): int(col) for tok, col in vec.vocabulary_.items()}
    idf = np.asarray(vec._tfidf.idf_, dtype=np.float64)
    booster = xgb.Booster()
    booster.load_model(_fetch("app/models/XGB_TFIDF.json")[0])
    return tiktoken.get_encoding("cl100k_base"), vocab, idf, booster


def _tfidf(codes, enc, vocab, idf):
    X = np.zeros((len(codes), len(idf)))
    for i, code in enumerate(codes):
        for tok in enc.encode(code, disallowed_special=()):
            col = vocab.get(tok)
            if col is not None:
                X[i, col] += 1
    X *= idf
    norm = np.linalg.norm(X, axis=1, keepdims=True)
    return X / np.where(norm == 0, 1, norm)


def score(codes):
    enc, vocab, idf, booster = _model()
    # Dense on purpose: the model was trained on .toarray() features, so zeros are values, not missing.
    return booster.predict(xgb.DMatrix(_tfidf(codes, enc, vocab, idf))).tolist()

In [ ]:
# %% 3. Released dataset (4 MB, no Ada embeddings) and repo-identical tokenization.
import json, pickletools, zipfile
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score
from sklearn.model_selection import train_test_split
from xgboost import XGBClassifier

W = "/content/oedingen"
os.makedirs(W, exist_ok=True)


def get(url, path, sha):
    if not os.path.exists(path):
        urllib.request.urlretrieve(url, path)
    assert hashlib.sha256(open(path, "rb").read()).hexdigest() == sha, path


get("https://th-koeln.sciebo.de/s/XZRR45yzO0rRuj3/download", f"{W}/data.zip",
    "804da451dabcd9311dfeb19a318e1b79f0d2d1e8719cde09a391f15607e34e93")
with zipfile.ZipFile(f"{W}/data.zip") as z:
    z.extractall(W)
df = pd.DataFrame([json.loads(l) for l in open(f"{W}/Unformatted_Balanced.jsonl")])  # file order = repo order
assert len(df) == 40158 and df.label.sum() == 20079 and df.id.nunique() == 6899
y = df.label.to_numpy()
ENC = _model()[0]
TOK = [ENC.encode(c) for c in df.code]  # repo default: raises on special tokens; none occur
ident = lambda x: x  # noqa: E731  (the repo's analyzer)


def report(y_true, p, pred):
    return dict(acc=accuracy_score(y_true, pred), prec=precision_score(y_true, pred),
                rec=recall_score(y_true, pred), f1=f1_score(y_true, pred),
                auc=roc_auc_score(y_true, p), fpr=pred[y_true == 0].mean())

In [ ]:
# %% 4. What the shipped artifact is. Expect True, True, ~1e-16, 0.0, then a tiny |dp| if XGB also saw all rows.
enc, vocab, idf, booster = _model()
full = TfidfVectorizer(analyzer=ident, max_features=1536)
X_full = full.fit_transform(TOK).toarray()
print("shipped vocab == fit on all 40,158 rows:", {int(k): int(v) for k, v in full.vocabulary_.items()} == vocab)
print("shipped idf   == fit on all 40,158 rows:", np.allclose(full.idf_, idf, rtol=0, atol=1e-12))
print("manual tfidf vs sklearn, max |diff|:", np.abs(_tfidf(df.code[:2000].tolist(), enc, vocab, idf) - X_full[:2000]).max())

get(_RAW + "Models/XGB_TFIDF.pkl", f"{W}/XGB_TFIDF.pkl",
    "79f2c384889791cbb6df1867da038a884ce210e51ca9afbf39e979428342222e")
blob = max((a for _, a, _ in pickletools.genops(open(f"{W}/XGB_TFIDF.pkl", "rb").read())
            if isinstance(a, (bytes, bytearray))), key=len)  # Booster raw bytes, read without unpickling
pkl_booster = xgb.Booster()
pkl_booster.load_model(bytearray(blob))
p_ship = np.asarray(score(df.code.tolist()))
p_pkl = pkl_booster.predict(xgb.DMatrix(X_full))
print("JSON vs pkl booster, max |dp|:", np.abs(p_ship - p_pkl).max())
p_refit = XGBClassifier().fit(X_full, y).predict_proba(X_full)[:, 1]
print("default XGB refit on all rows vs shipped (pkl saved by xgboost 2.0.3), max |dp|:",
      np.abs(p_refit - p_ship).max(), "label agreement:", ((p_refit > 0.5) == (p_ship > 0.5)).mean())
print("shipped, IN-SAMPLE (not a reproduction):", {k: round(v, 4) for k, v in report(y, p_ship, (p_ship > 0.5).astype(int)).items()})

In [ ]:
# %% 5. Reproduce Table 3 (XGB + TF-IDF, unformatted): repo recipe, 80/20 split by problem id, 10 seeds.
SEEDS = list(range(10))  # the paper does not name its seeds; the repo default is 42
problems = df["id"].unique()
rows = []
for seed in SEEDS:
    tr_ids, te_ids = train_test_split(problems, test_size=0.2, random_state=seed)
    tr, te = df.id.isin(tr_ids).to_numpy(), df.id.isin(te_ids).to_numpy()
    vec = TfidfVectorizer(analyzer=ident, max_features=1536)
    X_tr = vec.fit_transform([TOK[i] for i in np.flatnonzero(tr)]).toarray()
    X_te = vec.transform([TOK[i] for i in np.flatnonzero(te)]).toarray()
    clf = XGBClassifier().fit(X_tr, y[tr])
    rows.append({"seed": seed, "n_test": int(te.sum()),
                 **report(y[te], clf.predict_proba(X_te)[:, 1], clf.predict(X_te))})
    print(rows[-1])
res = pd.DataFrame(rows)
paper = {"acc": 98.28, "prec": 97.87, "rec": 98.70, "f1": 98.28, "auc": 99.84}
cmp = pd.DataFrame({"paper": paper, "ours_mean": res[list(paper)].mean() * 100,
                    "ours_sd": res[list(paper)].std(ddof=0) * 100})
cmp["delta_pp"] = cmp.ours_mean - cmp.paper
print(cmp.round(2))
print(f"miss rate: paper {100 - 98.70:.2f}%, ours {100 * (1 - res.rec.mean()):.2f}%; "
      f"FPR: paper ~2.14% (implied), ours {100 * res.fpr.mean():.2f}%")
ok = abs(cmp.delta_pp["acc"]) <= 1.0 and abs(cmp.delta_pp["rec"]) <= 1.0 and cmp.ours_mean["auc"] >= 99.5
print("REPRODUCED within tolerance" if ok else "NOT reproduced: see deltas")

In [ ]:
# %% 6. Smoke test of the module contract.
print(score(["n = int(input())\nprint(n * (n + 1) // 2)",
             "def solve():\n    n = int(input())\n    print(n * (n + 1) // 2)\n\nsolve()\n"]))